# DVH demo

# Imports

In [ ]:
import numpy as np, json
from pathlib import Path
from pymedphys._dvh.config import DVHConfig
from pymedphys._dvh.core import compute_dvh
from pymedphys._dvh.metrics import compute_metrics

Create temp directory for demo:

In [ ]:
demo_dir = Path("./dvh_demo")
demo_dir.mkdir(exist_ok=True, parents=True)

# Synthetic data: 64^3 grid, voxel 1 mm, dose = isotropic 3D Gaussian centred,
# sphere mask (radius 10 mm), fractional occupancy approximated as 0/1 for simplicity here.

In [ ]:
N = 64
voxel_mm = (1.0, 1.0, 1.0)
grid = np.arange(N) - (N - 1) / 2.0
X, Y, Z = np.meshgrid(grid, grid, grid, indexing="ij")
r2 = X**2 + Y**2 + Z**2
A = 1000.0  # cGy peak
sigma = 10.0  # mm
dose = A * np.exp(-r2 / (2 * sigma**2))

R = 10.0  # mm
mask = (r2 <= R**2).astype(np.float64)

In [ ]:
np.save(demo_dir / "dose.npy", dose)
np.save(demo_dir / "mask.npy", mask)

In [ ]:
cfg = DVHConfig(
    voxel_size_mm=(1.0, 1.0, 1.0),
    bin_width_cgy=1.0,
    min_dose_cgy=0.0,
    max_dose_cgy=float(np.max(dose)),
)
dvh = compute_dvh(dose, mask, cfg)
metrics = compute_metrics(dvh, dose_query_list_cgy=[200, 500, 800])

In [ ]:
with open(demo_dir / "dvh.json", "w") as f:
    json.dump({"dvh": dvh.to_dict(), "metrics": metrics}, f, indent=2)

In [ ]:
print("DVH total volume (cc):", dvh.total_volume_cc)
print(
    "Example metrics:", {k: metrics[k] for k in ["D95p_cGy", "D5p_cGy", "D0p03cc_cGy"]}
)